In [1]:
import xarray as xr
import numpy as np

# 1. 파일 경로 설정 (가지고 계신 nc 파일 이름으로 변경하세요)
#file_path = 'kim_r030_latlon (2).nc'
file_path = 'um_l015_latlon.nc'

print("NetCDF 파일을 불러오는 중입니다...")
ds = xr.open_dataset(file_path)

# (참고) 파일 내 변수 이름 확인하기
# 파일마다 위도/경도 변수명이 다를 수 있습니다 (예: lat/lon 또는 latitude/longitude)
# print(list(ds.variables.keys())) 

# 2. 위도(lat), 경도(lon) 2차원 배열 추출
# 만약 에러가 나면 위 주석을 해제하여 변수명을 확인하고 아래 'lat', 'lon'을 수정하세요.
lats = ds['lat'].values
lons = ds['lon'].values

print(f"격자망 로드 완료! 배열 크기: {lats.shape} (Y축, X축)")

# 3. 최단 거리 격자점(X, Y)을 찾는 함수
def find_nc_xy(target_lat, target_lon, lats_array, lons_array):
    # 위도 1도당 거리 차이를 보정하기 위한 상수 (제주 위도 33.3 기준)
    cos_lat = np.cos(np.radians(33.3))
    
    # 2차원 배열 전체에 대해 유클리디안 거리(제곱) 계산
    dist_sq = (lats_array - target_lat)**2 + ((lons_array - target_lon) * cos_lat)**2
    
    # 거리가 가장 작은(가장 가까운) 지점의 2차원 인덱스 찾기
    # 파이썬은 0부터 시작하므로 (y_idx, x_idx) 형태로 반환됩니다.
    min_idx = np.unravel_index(np.argmin(dist_sq, axis=None), dist_sq.shape)
    
    # 기상청 API는 X, Y 좌표를 1부터 시작하므로 각각 +1을 해줍니다.
    # 주의: NetCDF 배열은 보통 (Y축, X축) 순서로 저장되어 있습니다.
    api_Y = min_idx[0] + 1
    api_X = min_idx[1] + 1
    
    return api_X, api_Y

# ==========================================
# 4. 실제 지점 테스트
# ==========================================
target_locations = [
    {"name": "북쪽", "lat": 33.5260, "lon": 126.5860},
    {"name": "북동", "lat": 33.5580, "lon": 126.7480},
    {"name": "동쪽", "lat": 33.4620, "lon": 126.9360},
    {"name": "남동", "lat": 33.3250, "lon": 126.8320},
    {"name": "남쪽", "lat": 33.2380, "lon": 126.5140},
    {"name": "남서", "lat": 33.2060, "lon": 126.2900},
    {"name": "서쪽", "lat": 33.2950, "lon": 126.1620},
    {"name": "북서", "lat": 33.4650, "lon": 126.3200},
    {"name": "기존_ncm_남쪽", "lat": 33.3284, "lon": 126.8366},
    {"name": "기존_ncm_wind_동쪽", "lat": 33.5913, "lon": 126.7930},
    {"name": "기존_ncm_wind_서쪽", "lat": 33.4427, "lon": 126.1713}

]

print("\n--- 오프라인 격자점 변환 결과 ---")
for loc in target_locations:
    x, y = find_nc_xy(loc['lat'], loc['lon'], lats, lons)
    print(f"[{loc['name']}] 위경도({loc['lat']}, {loc['lon']}) -> API 호출용 격자: X={x}, Y={y}")

# 파일 닫기
ds.close()

NetCDF 파일을 불러오는 중입니다...
격자망 로드 완료! 배열 크기: (781, 602) (Y축, X축)

--- 오프라인 격자점 변환 결과 ---
[북쪽] 위경도(33.526, 126.586) -> API 호출용 격자: X=295, Y=87
[북동] 위경도(33.558, 126.748) -> API 호출용 격자: X=305, Y=90
[동쪽] 위경도(33.462, 126.936) -> API 호출용 격자: X=317, Y=83
[남동] 위경도(33.325, 126.832) -> API 호출용 격자: X=311, Y=73
[남쪽] 위경도(33.238, 126.514) -> API 호출용 격자: X=291, Y=66
[남서] 위경도(33.206, 126.29) -> API 호출용 격자: X=277, Y=64
[서쪽] 위경도(33.295, 126.162) -> API 호출용 격자: X=270, Y=70
[북서] 위경도(33.465, 126.32) -> API 호출용 격자: X=279, Y=83
[기존_ncm_남쪽] 위경도(33.3284, 126.8366) -> API 호출용 격자: X=311, Y=73
[기존_ncm_wind_동쪽] 위경도(33.5913, 126.793) -> API 호출용 격자: X=308, Y=92
[기존_ncm_wind_서쪽] 위경도(33.4427, 126.1713) -> API 호출용 격자: X=270, Y=81
